# Chain of Thought (CoT) 思维链推理 - 第一部分：基础与 Zero-shot

## 学习目标
1. 理解 CoT 的核心原理
2. 掌握 Zero-shot CoT 的使用
3. 了解基础数据结构

## 目录
1. [为什么需要 CoT](#1-为什么需要-cot)
2. [CoT 的数学原理](#2-cot-的数学原理)
3. [基础数据结构](#3-基础数据结构)
4. [Zero-shot CoT](#4-zero-shot-cot)
5. [练习](#5-练习)

In [ ]:
import sys
sys.path.insert(0, '..')

from src.chain_of_thought import (
    CoTStrategy, CoTStep, CoTExample, CoTResult,
    CoTPromptBuilder, ZeroShotCoT, FewShotCoT, AutoCoT,
)
print("模块加载成功！")

---
## 1. 为什么需要 CoT？

### 1.1 传统提示的问题

In [ ]:
# 传统提示 vs CoT 提示对比
print("="*60)
print("传统提示：")
print("-"*60)
traditional = """问题：商店有50个苹果，卖出23个，进货15个，卖出18个。现在有多少？
答案："""
print(traditional)
print("\n问题：LLM需要一步完成所有计算，容易出错")

print("\n" + "="*60)
print("CoT 提示：")
print("-"*60)
cot = """问题：商店有50个苹果，卖出23个，进货15个，卖出18个。现在有多少？

让我一步步思考：
1. 开始：50个
2. 卖出23个：50-23=27个
3. 进货15个：27+15=42个
4. 卖出18个：42-18=24个

答案：24个"""
print(cot)
print("\n优势：每步简单计算，可验证，可追溯")

### 1.2 CoT 的核心洞察

**Wei et al., 2022 的关键发现：**
- 分解复杂问题为简单子问题
- 生成的文本作为"工作记忆"
- 推理过程透明可解释
- 大模型(>100B)效果显著

In [ ]:
# 可视化对比
print("""
┌────────────────────────────────────────────────────────┐
│              传统提示 vs CoT 提示                       │
├────────────────────────────────────────────────────────┤
│ 传统: [问题] ──────────────────────────> [答案]         │
│              (黑盒，一步到位)                           │
│                                                        │
│ CoT:  [问题] ─> [步骤1] ─> [步骤2] ─> [步骤3] ─> [答案] │
│              (透明，逐步推理，可验证)                    │
└────────────────────────────────────────────────────────┘
""")

---
## 2. CoT 的数学原理

### 2.1 概率建模

**传统提示：** $P(answer|question) = P(a|q)$

**CoT提示：** $P(answer|question) = \sum_r P(a|r,q) \cdot P(r|q)$

其中 $r$ 是推理链

In [ ]:
# 直觉解释
print("""
为什么 CoT 更好？

假设3步推理，每步正确率90%：

传统方式（一步到位）：
  - 需同时做对3件事
  - 成功率 ≈ 0.9³ = 72.9%

CoT方式（逐步推理）：
  - 每步独立，可验证纠正
  - 简单子问题更容易做对
  - 错误可被后续步骤发现
  - 实际成功率更高
""")

### 2.2 三种主要变体

| 变体 | 描述 | 优点 | 缺点 |
|------|------|------|------|
| Zero-shot | 添加"让我们一步步思考" | 简单无需示例 | 效果不稳定 |
| Few-shot | 提供带推理链的示例 | 效果最好 | 需人工创建 |
| Auto-CoT | 自动生成示例 | 可扩展 | 质量依赖LLM |

---
## 3. 基础数据结构

### 3.1 CoTStrategy - 策略枚举

In [ ]:
print("可用的 CoT 策略：")
for strategy in CoTStrategy:
    print(f"  {strategy.name:20} -> '{strategy.value}'")

### 3.2 CoTStep - 推理步骤

In [ ]:
# 创建推理步骤
step1 = CoTStep(content="理解问题：计算15%的80", step_number=1, step_type="理解")
step2 = CoTStep(content="15% = 0.15", step_number=2, step_type="转换", confidence=0.95)
step3 = CoTStep(content="0.15 × 80 = 12", step_number=3, step_type="计算", confidence=0.99)

print("推理步骤：")
for step in [step1, step2, step3]:
    conf = f" (置信度:{step.confidence:.0%})" if step.confidence else ""
    print(f"  {step}{conf}")

In [ ]:
# 序列化
import json
print("序列化结果：")
print(json.dumps(step2.to_dict(), ensure_ascii=False, indent=2))

### 3.3 CoTExample - 示例

In [ ]:
example = CoTExample(
    question="小明有8本书，小红的书是小明的2倍，两人共有多少本书？",
    reasoning="""1. 小明有8本书
2. 小红有 8×2=16 本书
3. 共有 8+16=24 本书""",
    answer="24本书"
)

print("CoT示例：")
print(f"问题: {example.question}")
print(f"推理:\n{example.reasoning}")
print(f"答案: {example.answer}")

In [ ]:
# 格式化用于提示
print("格式化后：")
print(example.format())

In [ ]:
# 解析步骤
steps = example.parse_steps()
print(f"解析出 {len(steps)} 个步骤：")
for s in steps:
    print(f"  步骤{s.step_number}: {s.content}")

### 3.4 CoTResult - 推理结果

In [ ]:
result = CoTResult(
    question="15%的80是多少？",
    reasoning="15%=0.15, 0.15×80=12",
    answer="12",
    steps=[step1, step2, step3]
)

print(f"问题: {result.question}")
print(f"答案: {result.answer}")
print(f"步骤数: {result.num_steps}")

---
## 4. Zero-shot CoT

### 4.1 核心原理

**Kojima et al., 2022 发现：**
- 添加 "Let's think step by step" 即可激活推理
- 大模型(>100B)效果显著
- 无需任何示例

In [ ]:
# 创建 Zero-shot CoT
zero_shot = ZeroShotCoT(trigger="standard")

# 常用触发短语
print("常用触发短语：")
triggers = {
    "standard": "Let's think step by step.",
    "detailed": "Let's work through this step by step.",
    "decompose": "Let's break this down into steps."
}
for name, t in triggers.items():
    print(f"  {name}: {t}")

### 4.2 基本使用

In [ ]:
question = "商店有50个苹果，卖出23个后进货15个，现在有多少？"
prompt = zero_shot.get_prompt(question)

print("Zero-shot CoT 提示：")
print("="*50)
print(prompt)
print("="*50)

### 4.3 不同问题类型测试

In [ ]:
test_questions = [
    ("算术", "x + 5 = 12，x等于多少？"),
    ("逻辑", "所有猫是动物。小花是猫。小花是动物吗？"),
    ("多步", "100元买3本书每本15元，2支笔每支8元，剩多少？"),
]

print("不同问题类型：")
for q_type, q in test_questions:
    prompt = zero_shot.get_prompt(q)
    print(f"\n【{q_type}】{q}")
    print(f"  触发: ...{prompt[-40:]}")

### 4.4 Zero-shot CoT 局限性

In [ ]:
print("""
Zero-shot CoT 局限性：

1. 推理质量不稳定
   - 无示例引导，格式可能不一致
   - 复杂问题可能遗漏步骤

2. 领域适应性差
   - 专业领域效果有限
   - 需要领域特定推理模式

3. 小模型效果差
   - <100B参数效果不明显

适用场景：
  ✓ 快速原型验证
  ✓ 简单推理任务
  ✓ 没时间准备示例时
""")

---
## 5. 练习

### 练习 1：创建推理步骤

In [ ]:
# TODO: 为以下问题创建3个推理步骤
# 问题：一个长方形长10cm，宽6cm，周长是多少？

# step1 = CoTStep(...)
# step2 = CoTStep(...)
# step3 = CoTStep(...)

### 练习 2：创建 CoT 示例

In [ ]:
# TODO: 创建一个高质量的 CoT 示例
# 问题：火车3小时行驶240公里，平均速度是多少？

# my_example = CoTExample(
#     question="...",
#     reasoning="...",
#     answer="..."
# )

### 练习 3：使用 Zero-shot CoT

In [ ]:
# TODO: 使用 Zero-shot CoT 为以下问题生成提示
# 问题：如果今天是周三，5天后是周几？

# my_zero_shot = ZeroShotCoT(trigger="standard")
# my_prompt = my_zero_shot.get_prompt("...")
# print(my_prompt)

---
## 下一步

继续学习 **01b_ChainOfThought_FewShot.ipynb** 了解 Few-shot CoT